# 🚀 FaceKey Collector — Kaggle (Zero-Clutter 2-Cell Notebook)
### Dedicated Worker: `kaggle-worker-1` (4 Parallel CPU Workers, 1 Face Only, Turbo 150+ FPS)

> [!IMPORTANT]
> **CRITICAL KAGGLE SETTING: ACCELERATOR = NONE (CPU)**
> - **Save Your 30h GPU Quota**: MediaPipe Python runs on CPU SIMD (XNNPACK). Running data collection on a GPU notebook burns your weekly 30h GPU limit at 0% GPU load!
> - **Unlimited Free Hours**: Kaggle gives **100% UNLIMITED FREE CPU HOURS** (4 vCPUs, 30 GB RAM). In CPU mode, this collector runs 24/7 forever consuming **0 minutes of GPU quota**!
> - **Dual T4 GPU Allocation**: Save your 30h GPU quota for `5_Kaggle_Model_Trainer.ipynb` where PyTorch utilizes 100% of Dual T4 GPUs at 500+ FPS!

## ⚡ CELL 1: RUN DATA COLLECTOR (AUTO-SAVES TO CLOUD AFTER EVERY VIDEO)

In [ ]:
# ==============================================================================
# ⚡ CELL 1: START KAGGLE COLLECTOR (CONTINUOUS AUTO-SAVE & ZERO DISK BLOAT)
# ==============================================================================
import os, sys, shutil

# 1. Working directories
DRIVE_DIR = '/kaggle/working/FaceKeyDataset'
for folder in ('chunks', 'checkpoints', 'locks', 'data'):
    os.makedirs(f'{DRIVE_DIR}/{folder}', exist_ok=True)

# 2. Update to latest GitHub repository code (Self-Healing)
%cd /kaggle/working
APP_DIR = '/kaggle/working/FaceKeyAnimation'
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating FaceKeyAnimation to latest GitHub main...')
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print('[*] Fresh clone of FaceKeyAnimation from GitHub...')
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Fast Cloud Dependencies & Deno JS solver
!pip install -q -U --pre 'yt-dlp[default]' mediapipe opencv-python-headless numpy sounddevice mss av scipy huggingface_hub pysocks
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Dedicated Queue for Kaggle (Zero Overlap with Colab & Lightning)
urls_file = os.path.join(APP_DIR, 'sessions', 'youtube_urls_kaggle.txt')

# 5. Continuous Cloud Auto-Upload Configuration
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN and os.path.exists('/kaggle/working/hf_token.txt'):
    with open('/kaggle/working/hf_token.txt') as f:
        HF_TOKEN = f.read().strip()
if not HF_TOKEN:
    HF_TOKEN = input('Enter Hugging Face Token (or press Enter if saved): ').strip()
    if HF_TOKEN:
        with open('/kaggle/working/hf_token.txt', 'w') as f:
            f.write(HF_TOKEN)

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'

# 6. Launch 4-Worker CPU Collector (Continuous Auto-Save to Hugging Face Hub)
print('\n[*] Launching FaceKey Turbo 4-Worker Kaggle CPU Collector (kaggle-worker-1)...')
!python scripts/cloud_data_collector.py \
    --drive-dir '{DRIVE_DIR}' \
    --worker-id 'kaggle-worker-1' \
    --urls-file '{urls_file}' \
    --chunk-size 1 \
    --quality 480p \
    --num-workers 4 \
    --hf-token '{HF_TOKEN}' \
    --hf-repo '{HF_REPO}'


---
## 🚨 CELL 2: EMERGENCY SAVE & UPLOAD (RUN ANYTIME TO SAVE ALL DATA NOW)
> [!IMPORTANT]
> Run this cell if you stop the collector or before Kaggle session expires. It immediately packs all sessions on disk into chunks and uploads them to your cloud repository.

In [ ]:
# ==============================================================================
# 🚨 CELL 2: 1-CLICK INSTANT RESCUE & SAVE: PACK & UPLOAD ALL KAGGLE DATA NOW
# ==============================================================================
import os

# 1. Ensure latest code with auto-pack script
%cd /kaggle/working/FaceKeyAnimation
!git fetch origin main && git reset --hard origin/main

# 2. Automatically pack all sessions and upload all chunks to Hugging Face
!python scripts/emergency_hf_upload.py
